# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaitanyamadire01-dotcom/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. The Data Contract (5 Plain-Language Answers)

1. **What one row means (Grain):** One published URL/page for a given client over a monthly observation window.
2. **Which table(s) I will use:** The monthly aggregated page-level search performance table (`page_summary` / warehouse slice) partitioned by `month`.
3. **Which time window:** A mid-panel observation month (`month == '2026-03'`), leaving the final month (`2026-06`) strictly sealed as an out-of-sample holdout test partition.
4. **What I predict or rank (Target / Proxy):** Binary decay indicator (`is_declining = 1`) denoting a sustained drop in organic visibility over the subsequent window, prioritized by opportunity volume.
5. **One deliberate exclusion:** Pages with zero impressions (`impressions == 0`) or age under 30 days, as zero-impression URLs introduce extreme noise into CTR calculations and cannot realistically decay.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### 1. The Data Contract (5 Plain-Language Answers)

1. **What one row means (Grain):** One published URL/page for a given client over a monthly observation window.
2. **Which table(s) I will use:** The monthly aggregated page-level search performance table (`page_summary` / warehouse slice) partitioned by `month`.
3. **Which time window:** A mid-panel observation month (`month == '2026-03'`), leaving the final month (`2026-06`) strictly sealed as an out-of-sample holdout test partition.
4. **What I predict or rank (Target / Proxy):** Binary decay indicator (`is_declining = 1`) denoting a sustained drop in organic visibility over the subsequent window, prioritized by opportunity volume.
5. **One deliberate exclusion:** Pages with zero impressions (`impressions == 0`) or age under 30 days, as zero-impression URLs introduce extreme noise into CTR calculations and cannot realistically decay.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Read Hugging Face token from environment
token = os.environ.get("HF_TOKEN")

try:
    from datasets import load_dataset
    print("Loading mid-panel slice (month=2026-03) from Hugging Face...")
    ds = load_dataset("FlyRank/internship-warehouse", split="train", token=token)
    df_raw = ds.to_pandas()
    if 'month' in df_raw.columns:
        df_mid = df_raw[df_raw['month'] == '2026-03'].copy()
    else:
        df_mid = df_raw.copy()
    print("Successfully loaded slice from warehouse.")
except Exception as e:
    print(f"Fallback to starter CSV (reason: {e})")
    candidate_paths = [
        Path("data/raw/content_refresh_anonymized.csv"),
        Path("../../data/raw/content_refresh_anonymized.csv"),
        Path("../data/raw/content_refresh_anonymized.csv"),
    ]
    path = next(p for p in candidate_paths if p.exists())
    df_mid = pd.read_csv(path)

# ---------------------------------------------------------
# Fact 1: Grain (Verify (client_id, url) uniqueness)
# ---------------------------------------------------------
url_col = 'url' if 'url' in df_mid.columns else ('page_id' if 'page_id' in df_mid.columns else df_mid.columns[1])
is_unique = df_mid.duplicated(subset=['client_id', url_col]).sum() == 0
total_rows = len(df_mid)
unique_pairs = df_mid[['client_id', url_col]].drop_duplicates().shape[0]

print("\n=== FACT 1: GRAIN ===")
print(f"Grain definition: (client_id, {url_col})")
print(f"Total rows: {total_rows:,} | Unique pairs: {unique_pairs:,}")
print(f"Is grain strictly unique per row? {is_unique}")

# ---------------------------------------------------------
# Fact 2: Row Count & Date Span
# ---------------------------------------------------------
print("\n=== FACT 2: ROW COUNT & SPAN ===")
print(f"Row count for slice: {total_rows:,} rows")
if 'month' in df_mid.columns:
    print(f"Month partition: {df_mid['month'].unique()}")
elif 'date' in df_mid.columns:
    print(f"Date Span: {df_mid['date'].min()} to {df_mid['date'].max()}")
else:
    print(f"Content age span: {df_mid['content_age_days'].min()} to {df_mid['content_age_days'].max()} days")

# ---------------------------------------------------------
# Fact 3: Availability (Filter with IS TRUE / Active impressions)
# ---------------------------------------------------------
print("\n=== FACT 3: AVAILABILITY ===")
surviving = (df_mid['impressions_90d'] > 0).sum()
survival_pct = (surviving / total_rows) * 100
print(f"Rows meeting availability filter (impressions_90d > 0): {surviving:,} / {total_rows:,} ({survival_pct:.1f}%)")

Fallback to starter CSV (reason: No module named 'datasets')

=== FACT 1: GRAIN ===
Grain definition: (client_id, client_id)
Total rows: 30,000 | Unique pairs: 32
Is grain strictly unique per row? False

=== FACT 2: ROW COUNT & SPAN ===
Row count for slice: 30,000 rows
Content age span: 90 to 564 days

=== FACT 3: AVAILABILITY ===
Rows meeting availability filter (impressions_90d > 0): 30,000 / 30,000 (100.0%)


In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

# Filter dataset to available rows
df_features = df_mid[df_mid['impressions_90d'] > 0].copy()
y = (df_features['trend_direction'].str.lower() == 'down').astype(int)

# 1. Five honest features (all knowable prior to outcome)
df_features['is_top_page'] = (df_features['avg_position'] <= 10.0).astype(int)
honest_cols = ['avg_position', 'impressions_90d', 'ctr', 'content_age_days', 'is_top_page']
X_honest = df_features[honest_cols]

# Evaluate honest model
X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.3, random_state=42)
clf = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
honest_p = precision_score(y_te, clf.predict(X_te), zero_division=0)

print("=== 5 HONEST FEATURES ===")
print("1. avg_position: Knowable at decision moment (measured over prior 90 days).")
print("2. impressions_90d: Knowable at decision moment (measured over prior 90 days).")
print("3. ctr: Knowable at decision moment (prior clicks / impressions).")
print("4. content_age_days: Knowable at decision moment (metadata publication date).")
print("5. is_top_page: Knowable at decision moment (boolean flag from past position <= 10).")
print(f"\nHonest Model Precision: {honest_p:.4f}")

# 2. The Trap: Deliberate label leakage
df_features['LEAKED_target_proxy'] = y + np.random.normal(0, 0.05, size=len(df_features))
X_leaked = df_features[honest_cols + ['LEAKED_target_proxy']]
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaked, y, test_size=0.3, random_state=42)

clf_leaked = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr_l, y_tr_l)
leaked_p = precision_score(y_te_l, clf_leaked.predict(X_te_l), zero_division=0)

print("\n=== THE LEAKAGE TRAP ===")
print("Added feature: 'LEAKED_target_proxy' (derived directly from future outcome label).")
print(f"Artificially inflated Precision: {leaked_p:.4f}")

# 3. Drop the leaked feature and retain honest baseline
df_features.drop(columns=['LEAKED_target_proxy'], inplace=True)
print(f"\nRemoved leaked column. Preserved honest baseline Precision: {honest_p:.4f}")

=== 5 HONEST FEATURES ===
1. avg_position: Knowable at decision moment (measured over prior 90 days).
2. impressions_90d: Knowable at decision moment (measured over prior 90 days).
3. ctr: Knowable at decision moment (prior clicks / impressions).
4. content_age_days: Knowable at decision moment (metadata publication date).
5. is_top_page: Knowable at decision moment (boolean flag from past position <= 10).

Honest Model Precision: 0.6151

=== THE LEAKAGE TRAP ===
Added feature: 'LEAKED_target_proxy' (derived directly from future outcome label).
Artificially inflated Precision: 1.0000

Removed leaked column. Preserved honest baseline Precision: 0.6151


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Identified Limitation:

The slice aggregates impressions and clicks over a 90-day rolling window. While this smooths out weekend and seasonal volatility, it introduces a temporal lag: sudden algorithmic shifts or technical indexing issues in the final 7–14 days are obscured and will not be detected until traffic decay has already occurred.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Five plain-words contract answers filled.
- [x] Three verification queries executed with outputs visible.
- [x] Five honest features built with "knowable because" explanations.
- [x] Deliberate leakage experiment demonstrated and reverted.
- [x] One named limitation documented.